<a href="https://colab.research.google.com/github/mtfehl/BSE/blob/main/11_foundations_regression_in_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src = "https://github.com/barcelonagse-datascience/academic_files/raw/master/images/BSE_DSC_HEADER.jpg">

$\newcommand{\bb}{\boldsymbol{\beta}}$
$\DeclareMathOperator{\Gau}{\mathcal{N}}$
$\newcommand{\bphi}{\boldsymbol \phi}$
$\newcommand{\bx}{\boldsymbol{x}}$
$\newcommand{\bu}{\boldsymbol{u}}$
$\newcommand{\by}{\boldsymbol{y}}$
$\newcommand{\whbb}{\widehat{\bb}}$
$\newcommand{\hf}{\hat{f}}$
$\newcommand{\tf}{\tilde{f}}$
$\newcommand{\ybar}{\overline{y}}$
$\newcommand{\E}{\mathbb{E}}$
$\newcommand{\Var}{Var}$
$\newcommand{\Cov}{Cov}$
$\newcommand{\Cor}{Cor}$

In [ ]:
if (!require('glmnet')) install.packages("glmnet") #Key package(see later), run now, it takes a while
library("glmnet")

# Statistical, Supervised and Unsupervised Learning

**Statistical learning** is about learning a quantity of interest from some source of data. Broadly speaking, we focus on two kinds of statistical learning:

1. **Supervised Learning**: The goal here is to use some *inputs* to predict an *output*. We observe both *inputs* and *outputs*, so we can *learn* from the examples we have, hence supervised learning.

2. **Unsupervised Learning**: Learning *without* a teacher, or correct examples to learn from. The goal is typically to uncover low-dimensional structure in a high-dimension vector.

# Supervised Learning: prediction
##  Summary and problem set up

We are interested in predicting a target variable, $y \in \mathbb{R}$, using information from a vector of predictor variables, $\mathbf{x} = (x_1,\dots  ,x_p)' \in \mathbb{R}^p$. We may think that the prediction question can be formalized thinking that there exists a function $f$, such that
$$ y = f(\mathbf{x}) + u,$$
where $u$ is a _noise_ independent of $f$ and that it has zero mean.

This choice justifies using $f$ to predict $y$ given $\mathbf{x}$. Our predictions need to be _scored_ by some loss function. A typical choice is the square loss. The _expected (square) loss_, or the _mean squared error_ of a forecast $f({\bf x})$ is given by

$$ \text{MSE} = \E\Big[ \Big( y - f({\bf x}) \Big)^2\Big] $$

**First key idea is that we judge an algorithm/a model based on how well it predicts y, i.e. what we can observe**

It can be easily verified that the function that minimizes this particular loss function is $\E[ y | {\bf x}]$ , the conditional expectation of $y$ given ${\bf x}$.

 - Typically, however, we do not know what $\E[y | {\bf x}]$ looks like, and we need to _learn_, or _estimate_ it from the data. A natural starting point is to _assume_ this function can be reasonably approximated by a linear function.
 - Another big issue is that we do not know how to compute $\text{MSE}$ (question: why?)

To fix ideas, we will start from the $p=1$ case and move to the large $p$ case. We will discuss model selection and evaluation, the bias-variance tradeoff, model stability, overfitting and regularization.



## Read the data

We will start studying an artificial dataset to understand the methodology. We will move to real data and applications as we move along.

In [ ]:
set.seed(1)
x <- seq(0,1,1/9)
y <- sin(x*2*pi)+rnorm(length(x),sd=0.1)
y[4] <- sin(x[4]*2*pi)+rnorm(1,sd=0.2) #modify one data point with an outlier
data <- data.frame(x=x,y=y)
data

In two dimensions $(p=1)$ we can easily visualize our data.

In [ ]:
# Simple plot to get a feel for the relationship between x and y.
plot(data)
# or
plot(data$x, data$y)

## Our first learning function: linear in $\bf x$   
We are interested in predicting $y_i$ given a sample of size $i=1,\dots,n$ and some predictor variables $\bf{x}_i$.
In general, we are interested in finding the functional form of the conditional expectation of $y$ given $x$:

$$ y_i = f(\mathbf{x}_i , \bb) + u_i ,$$

and we call $f(\mathbf{x}_i , \bb)$  the *learning* or *regression function* . The regression function typically depends on the predictors, $\mathbf{x}_i$ and on some typically unknown parameters, $\bb$.

We begin by considering learning functions that are linear in the parameters ($\bb$) and in the predictors ($x_i$'s):
$$ y_i = \beta_0 + \sum_{i=1}^{p}x_i\beta_i + u_i $$
or, equivalently,
$$ y_i = {\bf{x}}_i' {\bb} + u_i $$
where ${\bf{x}_i} = (1, x_{i\,1},\dots,x_{i\,p})'$ and $\bb = (\beta_0,\dots,\beta_{p})'$.

## Estimating our linear function: Least Squares

A typical way to _learn_ the parameters of the model is to estimate them based on some data. Typically, we will solve
$$ \hat \bb = \arg\min_{\bb} \sum_{i=1}^n\Big( y_i - \bx_i' \bb \Big)^2 $$

where $n$ is the number of observations in our sample.

Hence, we can state that the linear form implied by the parameters $\hat \bb$ is the one that best fits our data cloud in the least square sense: In two dimensions, it is the line for which the squared deviations are minimal.

Minimizing least squares is a very simple problem. In particular, with some calculus, we can find closed form solutions for $\hat \bb$, or at least express it as the solution to a linear system of equations.

In nice enough environments and denoting by $\bf X$ and $\bf y$ the $n\times p$ matrix of $\bf{x}_1,\dots, \bf{x}_n$ and the vector of $y_1,\dots, y_n$, the closed form solution is $\hat \bb = (\bf{X}'\bf{X})^{-1}(\bf{X}'\bf{y})$.

Moreover, under some conditions, the solution to the least squares problem is equivalent to that obtained by maximum likelihood (for example, under gaussianity of $u_i$).

Remark on notation: bold-face for vectors, otherwise scalars; bold-face capital letters for matrices

Lets see how to estimate a linear model in R.


In [ ]:
reg <- lm(y~x,data=data)
reg

In [ ]:
summary(reg)

In [ ]:
attributes(reg)

In [ ]:
reg$coefficients

We could of course solve it using linear algebra:

In [ ]:
X  <- cbind(1,data$x)
y  <- cbind(data$y)
X

In [ ]:
solve(t(X)%*%X, t(X)%*%y)

### Predicting new data

Generally speaking, we are interested in creating predictions for which we have no observed $y$'s.
We want a forecast of inflation for tomorrow, not yesterday.
In other words, we want to learn the function so that we can make predictions on data that is yet unseen.

Given our estimates $\hat \bb$ (from the regression), we will compute $f( \bx_o , \hat{\bb} )$, for some unseen $\bx_o$.

In [ ]:
# create the new, unseen, points
x_predict <- seq(0,1,0.01)
x_predict

We can do it manually:

In [ ]:
yhat_predict  <- cbind(1, x_predict) %*% solve(t(X)%*%X, t(X)%*%y)
yhat_predict[1:10]

Or we could use R functions:

In [ ]:
x_predict  <- cbind.data.frame(x=x_predict)
x_predict[1:10,]

In [ ]:
y_hat <-  predict(reg, newdata = x_predict)

What should we expect from this exercise? Well, we estimated the parameters of a linear model, so we expect our predictions to be _linear_ in X_new.

In [ ]:
plot(x_predict[,1],y_hat,t='l',col='firebrick',lwd=2,ylim=c(-1,1), ylab='y',xlab='x')
points(data,col='orange',pch=19,cex=1.5)
grid()
legend('topright',c('Training Data','Prediction Data'), col = c('orange','firebrick'),
       lty=c(NA,1), pch=c(19,NA),lwd=4)
# pts are f(x) - line is f_hat(x0)

This is the line that best fits our data...

## A more flexible learning function: linear in parameters and features, nonlinear in input

Clearly, a linear regression function is not able to capture the patterns in our curve data. We need a more flexible learning function:


$$ y_i = \beta_0 + \beta_1 x_i + \beta_2 x_i^2 + \ldots +\beta_n x_i^{p} $$

We have added to our linear model _nonlinear_ transformations of our input $x$.

This is a constructive perspective discussed already in the *Feature engineering notebook*: we are creating **features**,  new input variables that are transformations of the original ones. In the above construction, if we let the vector of features for the $i$th data point be

$$ {\bf  F}_i =(1,x_i,x_i^2,\ldots,x_i^{p})'$$

then

$$ f(x_i,\bb) =  {\bf  F}_i ' \bb$$

Notice that we now have $p+1$ predictors, even though $x$ is 1-dimensional. The choice of polynomial features is simply for illustration.

In [ ]:
p  <- 3
x_poly  <- sapply(0:p, function(w) data$x^w)
x_poly

Lets call the dataset obtained by constructing the 3rd degree polynomials `data_med`.

In [ ]:
colnames(x_poly) <- paste0('p',0:p)
data_med  <- cbind.data.frame(y=data$y,x_poly)
data_med

And the corresponding regression model `reg_med`.

In [ ]:
reg_med  <- lm(y ~ 0 + . ,data=data_med)
summary(reg_med)

In [ ]:
x_poly_predict  <- cbind.data.frame(sapply(0:p, function(w) x_predict$x^w))
colnames(x_poly_predict) <- paste0('p',0:p)
y_hat_med  <- predict(reg_med,newdata = x_poly_predict)
head(y_hat_med)

In [ ]:
plot(x_poly_predict$p1,y_hat_med,t='l',col='firebrick',lwd=2,ylim=c(-1,1), ylab='y',xlab='x')
points(data,col='orange',pch=19,cex=1.5)
lines(data_med$p1,reg_med$fitted.values,col='darkblue',lwd=3)
grid()
legend('topright',c('Training Data','Prediction Data','Fitted Values'), col = c('orange','firebrick','darkblue'),
       lty=c(NA,1,1), pch=c(19,NA,NA),lwd=4)

Our more flexible polynomial is able to better capture the dynamics of our data.
Adding 3rd degree polynomials really helped, it seems. Maybe we should try to add higher degree polynomials?


In [ ]:
p  <- 9
x_poly_large  <- sapply(0:p, function(w) data$x^w)
colnames(x_poly_large) <- paste0('p',0:p)
data_large  <- cbind.data.frame(y=data$y,x_poly_large)
data_large

In [ ]:
reg_large  <- lm(y ~ 0 + . ,data=data_large)
summary(reg_large)

In [ ]:
x_large_predict  <- cbind.data.frame(sapply(0:p, function(w) x_predict$x^w))
colnames(x_large_predict) <- paste0('p',0:p)
y_hat_large  <- predict(reg_large,newdata = x_large_predict)
head(y_hat_large)

In [ ]:
plot(x_large_predict$p1,y_hat_large,t='l',col='firebrick',lwd=2,ylim=c(-1,1.4), ylab='y',xlab='x')
points(data,col='orange',pch=19,cex=1.5)
lines(data_large$p1,reg_large$fitted.values,col='darkblue',lwd=3)
grid()
legend('topright',c('Training Data','Prediction Data','Fitted Values'), col = c('orange','firebrick','darkblue'),
       lty=c(NA,1,1), pch=c(19,NA,NA),lwd=4)

In [ ]:
dim(data_large)

## Two problems emerge

+ It looks like we are _overfitting_ our data.
If we have 10 data points and 10 parameters to estimate, we will quite clearly fit perfectly in sample (the blue line perfectly lines up with the orange points).
However, _in between_ our data points, we may not do a good job at predicting.
But could we make this notion more formal?
+ As we come up with more features ($p$ increases), we start having a lot of issues
  - we may lose the possibility of even doing OLS ($p>n$ scenario)
  - the statistical properties of my estimates deteriorates and/or may require completely no tools. Standard theory does not apply anymore (see field of High-dimensional Statistics)

# Model evaluation

There are many ways to evaluate the quality of our model.
Perhaps the most standard is the **R^2**. We can write it as follows

$$ \text{R-squared} = 1 - {\sum_{i=1}^n (y_i - {\bf x}_i'\hat\bb)^2 \over \sum_{i=1}^n (y_i - \ybar)^2}= 1 - {SSR \over TSS}$$

Clearly, large R-squared is equivalent to a small **sum of squared residuals** (SSR: sum of squared residuals, TSS: total sum of squares). The better our models, the smallest the sum of squared residuals.

One can show that **R^2** is related to the **square of the correlation coefficient** between our predictions and the realized values (if we include the intercept). This squared correlation coefficient is used so frequently that it has a name: R-squared.

Lets plot our data and the predictions given by the least squares solution.


In [ ]:
options(repr.plot.width=16, repr.plot.height=8)
par(mfrow=c(1,3))
plot(y,reg$fitted.values,pch=19,col='darkblue',cex=5.5,ylab='fitted values',xlab='realized values',
    ,main=sprintf('p=1, R2: %.2f',cor(y,reg$fitted.values)^2))
lines(y,y,lwd=3,col='firebrick')
plot(y,reg_med$fitted.values,pch=19,col='darkblue',cex=5.5,ylab='fitted values',xlab='realized values'
     ,main=sprintf('p=3, R2: %.2f',cor(y,reg_med$fitted.values)^2))
lines(y,y,lwd=3,col='firebrick')
plot(y,reg_large$fitted.values,pch=19,col='darkblue',cex=5.5,ylab='fitted values',xlab='realized values'
     ,main=sprintf('p=9, R2: %.2f',cor(y,reg_large$fitted.values)^2))
lines(y,y,lwd=3,col='firebrick')

What is happening? We are computing the **in-sample** fit. As we add more parameters, our model becomes more flexible and the **in-sample** fit increases. In fact, when we have $p=n$, we achieve perfect fit. So the insample R-squared is _too optimistic_.

In general, we are interested in predicting points that we have not yet seen, not predicting perfectly the points we have already seen. We need the R-squared to be informative of how our model would perform in unseen data.

*Exercise*: try to do the same computing the MSE error? Does this metric change the way you are evaluating your models?

## Judging the models based on "fresh new" data


Let $x^*$ a point at which we want to obtain the prediction. We train several models based on  *training data* $(x_i,y_i),i=1,\ldots,n$, we are interested in reliable estimates of

+ Mean Squared Error (MSE):

$$ \text{MSE} = \E\Big[\Big(Y - {\bf x^*}'\hat\bb\Big)^2\Big], \text{ and }$$

+ Squared correlation between $y^*$ and $\hf_n(\bx^*)$:

$$ R^2 = \Cor(Y,{\bf x^*}'\hat\bb)^2$$

where in both quantities the expectations of wrt to the "true data generating distribution" (important: this generally includes both $y$ and $\mathbf{x}$).

Up to now, we estimated the population expectation with the sample averages over the training data. For example, in the case of MSE, we used
$$\widehat{MSE} =\sum_{i=1}^n (y_i -x_i'\hat{\beta})^2.$$

- The use of this estimator is justified by the SLLN

- However, we just saw that the insample version of these objects is too optimistic by construction. Why?
  + We are **using the data-twice**: to estimate $\beta$ and then to estimate $MSE$.

## How to move forward?

*Three ideas*

1) Sample splitting: divide your data set into training and test data sets

2) Information criteria

3) Cross-validation

1),2) and 3) are all ways to estimate **out-of-sample quantities** (e.g. $MSE$, out-of-sample $R^2$, etc.)

## 1) Sample split

If you have a large sample size, you can split the the data into two parts

+ **Training data**: you take a fraction of $n$ to estimate $\hat{\beta}$
+ **Test data**: you use the remaining fraction to estimate $MSE$, or other out-of-sample quantities

In [ ]:
#In the example below, we did not have a large sample.
# But since we are dealing with a synthtetic data, we can generate extra data
x_test=seq(0,1,0.05)
y_test=sin(x_test*2*pi)+rnorm(length(x_test),sd=0.1)

In [ ]:
x_small_test <- cbind.data.frame(x=x_test)
colnames(x_small_test) <- 'x'
x_med_test <- cbind.data.frame(sapply(0:3, function(w) x_test^w))
colnames(x_med_test) <- paste0('p',0:3)
x_large_test <- cbind.data.frame(sapply(0:9, function(w) x_test^w))
colnames(x_large_test) <- paste0('p',0:9)
#Let's try to use the three models to predict the y_test
yhat_test_small <- predict(reg,newdata=x_small_test)
yhat_test_med <- predict(reg_med,newdata=x_med_test)
yhat_test_large <- predict(reg_large,newdata=x_large_test)

In [ ]:
#Compute the an estimate of the MSE
mse_small <- mean((yhat_test_small-y_test)^2)
mse_med <- mean((yhat_test_med-y_test)^2)
mse_large <- mean((yhat_test_large-y_test)^2)
print(c(mse_small,mse_med,mse_large))

Here, we recover a result more aligned with our intution.

Note: none of these models is correct though!

**Sample split rule**:
- no general rule, but you have 70-30,80-20 as very common
- you are balancing off the estimation error in the two steps

Possibly, **sample split** is the best thing you can do if you have the luxury to do it

## 2) Information criteria

Information criteria rely on a statistical model (often Gaussian) to find a way to estimate $MSE$.

The general structure is

`Information Criteria= - log likelihood at MLE + penalty #parameter`

You have several:

- Akaike Information criterion
- Bayesian Information criterion
- etc.

All these are estimators for MSE. In general, the lower the better

In [ ]:
#AIC
print(c(AIC(reg),AIC(reg_med),AIC(reg_large)))
#BIC
print(c(BIC(reg),BIC(reg_med),BIC(reg_large)))

General comments

- All information criteria are at best an approximation
- For some problem, some are better than others (e.g. AIC is known to be pretty poor to decide important variables)
- Big advantage wrt to data split: no split neeeded!
- Still, it is clear that it is suboptimal in comparison to data split to choose "best model"

## 3) Cross-validation

The third strategy is hybrid: you are in a situation where the data are scarce (small "n") but you want to do better than information criteria.

**Cross-validation** iterates through various split of your data


<img src='https://github.com/barcelonagse-datascience/academic_files/raw/master/images/cv.png'>

(Picture from medium)

Different ways of doing the split.

Let's consider the **leave-one-out cross-validation estimator**. The intuition is very simple.
Instead of using **all** our data to estimate the model, we:

+ Use all data points but the $i$th to estimate $\hat \bb_{-i}$

+ Using the estimated $\hat \bb{-i}$, compute the prediction for the ("unseen") $i$th training data point:

$$ {\bf x}_i \hat \bb_{-i}$$

+ Estimate the MSE or $R^2$ on the _hold out_ sample (in this case, the $i$th observation)

$$(y_i - {\bf x}_i \hat \bb_{-i})^2$$

+ We can do this for each data point $i$ and then average the estimates:

$${1 \over n} \sum_{i=1}^n (y_i - {\bf x}_i \hat \bb_{-i})^2$$

We can implement this ourselves in many ways. First the longest:

In [ ]:
#recall data_large is the dataframe with the original dataset (only training + many predictors)
T  <- nrow(data)
cvp_pred  <- cvp_med_pred  <- cvp_large_pred  <- rep(NA,T)
for( t in 1:T ){
    # Define train and test data
    train_data  <- data_large[-t,]
    test_data  <- data_large[t,]
    # Estimate model on train
    cvp_reg  <- lm(y ~ 0 + p0 + p1, data=train_data)
    cvp_med_reg  <- lm(y ~ 0 + p0 + p1 + p2 + p3, data=train_data)
    cvp_large_reg  <- lm(y ~ 0 + ., data=train_data)
    # Predict on testing data
    cvp_pred[t]  <- predict(cvp_reg,test_data)
    cvp_med_pred[t]  <- predict(cvp_med_reg,test_data)
    cvp_large_pred[t]  <- predict(cvp_large_reg,test_data)
}

In [ ]:
#compute LOOCV MSE
cv_mse_small <- mean((cvp_pred-data$y)^2)
cv_mse_med <- mean((cvp_med_pred-data$y)^2)
cv_mse_large <- mean((cvp_large_pred-data$y)^2)
print(c(cv_mse_small,cv_mse_med,cv_mse_large))

In [ ]:
#now, the LOOCV R^2
par(mfrow=c(1,3))
plot(y,cvp_pred,pch=19,col='darkblue',cex=1.5,ylab='fitted values',xlab='realized values',
    ,main=sprintf('p=1, LOOCV R2: %.2f',cor(y,cvp_pred)^2))
lines(y,y,lwd=3,col='firebrick')
plot(y,cvp_med_pred,pch=19,col='darkblue',cex=1.5,ylab='fitted values',xlab='realized values'
     ,main=sprintf('p=3, LOOCV R2: %.2f',cor(y,cvp_med_pred)^2),ylim=c(-10,10))
lines(y,y,lwd=3,col='firebrick')
plot(y,cvp_large_pred,pch=19,col='darkblue',cex=1.5,ylab='fitted values',xlab='realized values'
     ,main=sprintf('p=9, LOOCV R2: %.2f',cor(y,cvp_large_pred)^2))
lines(y,y,lwd=3,col='firebrick')

In [ ]:
plot(data$x,cvp_large_pred,t='l',col='firebrick',lwd=2,ylim=c(-2,2), ylab='y',xlab='x',lty=5)
lines(data$x,cvp_med_pred,t='l',col='darkblue',lwd=2,ylim=c(-1,1.4), ylab='y',xlab='x',lty=1)
lines(data$x,cvp_pred,t='l',col='black',lwd=2,ylim=c(-1,1.4), ylab='y',xlab='x',lty=2)
points(data,col='orange',pch=19,cex=1.6)
legend('topright',c('Data','p=9','p=3','p=1'), col = c('orange','firebrick','darkblue','black'),
       lty=c(NA,5,1,2), pch=c(19,NA,NA,NA),lwd=4)
grid()

Again, the Cross-Validated estimates differ from the insample estimates. It becomes very clear that the heavily parametrized model is very good insample, but very bad out of sample.
Simple models may have a smaller R-squared in sample, but their performance is more stable.

General comments:
- Note the only way of doing CV: leave-k-out CV, etc.
- Still these estimates have some form of dependence: there is a bias that sample split estimate don't have.
- *Computationally is very heavy*
  + many model to refit (now, we have considered three, but in practice you would have many more)
  + leave-k-out is even more computationally expensive


This leads us into the next topic:

## The bias-variance tradeoff

Suppose that our data generating process is additive, i.e. :$ y = f(\mathbf{x}) + u$, where $u\sim D(0,\sigma^2_u)$ and that we predict using $\hat{f}(x)$. Then, conditional on a value of $x_0$, We are interested in:

\begin{align}
\text{MSE} & = \E\Big[\Big(\hat{f}(x_0) - y \Big)^2  \Big] \\
           & = \E\Big[\Big(\hat{f}(x_0) - f(x_0) - u  \Big)^2\Big] \\
           & = \sigma^2_u +  \E\Big[\Big(\hat{f}(x_0) - {f}(x_0)\Big)^2 \Big] \\
           & = \sigma^2_u +  \E\Big[\Big(\hat{f}(x_0) - \E[\hat{f}(x_0)]  + \E[\hat{f}(x_0)] - {f}(x_0)\Big)^2 \Big | x = x_0\Big] \\
           & = \sigma^2_u +  \E\Big[\Big(\hat{f}(x_0) - \E[\hat{f}(x_0)]\Big)^2\Big]  +
                             \E\Big[\Big(\E[\hat{f}(x_0)] - {f}(x_0)\Big)^2 \Big] + \\
                            & ~~~ 2\E\Big[\Big(\E[\hat{f}(x_0)] - {f}(x_0)\Big)\Big(\hat{f}(x_0) - \E[\hat{f}(x_0)]\Big) \Big] \\
           & = \sigma^2_u +  \E\Big[\Big(\hat{f}(x_0) - \E[\hat{f}(x_0)]\Big)^2 \Big]  +
                             \E\Big[\Big(\E[\hat{f}(x_0)] - {f}(x_0)\Big)^2 \Big] + \\
                            & ~~~ 2\Big(\E[\hat{f}(x_0)] - {f}(x_0)\Big)\E\Big[\Big(\hat{f}(x_0) - \E[\hat{f}(x_0)]\Big) \Big] \\
           & = \underset{\text{irreducible error }}{\sigma^2_u} +
           \underset{\text{variance of $\hat{f}$}}{\E\Big[\Big(\hat{f}(x_0) - \E[\hat{f}(x_0)]\Big)^2\Big]}  + \underset{\text{bias sq.}}{\Big(\E[\hat{f}(x_0)] - {f}(x_0)\Big)^2 }
\end{align}


+ *Procedures* that use *few degrees of freedom* are stable but the learning function they estimate can be systematically far off from the optimal one (**bias**). They would have comparable R-squared and leave-one-out CV R-squared


+ *Procedures* that use *many degrees of freedom* are overly sensitive to training data (which induces higher **variance**) but their flexibility allows them to approximate well the target series, and reduce bias. They would have near-1 R-squared and near-0 leave-one-out CV R-squared

This opens a lot of possibilities! In particular, we can trade-off bias and variance.

The following figure, taken from Bishop, shows the estimated root mean squared error - blue is in-sample, red is analogous to leave-one-out CV - for increasing values of $p$ (denoted by $M$ in the fig).

<img src="https://github.com/barcelonagse-datascience/academic_files/raw/master/images/bishop_overfit.png">

We want **algorithms that can strike a good bias-variance tradeoff**!

The remaining of this lecture is devoted to:

1. Studying such algorithms: e.g. the LASSO; they use the same linear-in-features model but a different loss function
2. Discussing how to estimate the MSE from data; we will revisit CV.

## Summary up to here

- In modern applications, we could have many features (large "p"). Maybe because I have many variables I wanna consider (big data), or because I consider many possible transformations
- Large "p" is a world of troubles:
 + High-risk of overfitting
 + OLS is not defined when $p>n$
 + We lose statistical properties
 + We did not touch it so far but **we also lose interpretability** (too many varibles in the model, how can I make sense of it?)

- A predictive focus in judging the model rescue us to an extent

- Still many unresolved problems
  + $p>n$
  + Many model fit to perform and model to compare: very computationally expensive
  + Judging the model based on prediction is nice but we are still relying on a lot of assumptions
  + Interpretability still unresolved


  What's the way forward?

# Sparsity and shrinkage methods

**Assumption**: there is an underlying low dimensional model. This can be found in two ways

+ Dimension reduction (PCA, etc.)
+ Model selection (**sparsity**): out of the p predictors, only a small (small relative to?) actually matters.

We focus for now on the second assumption.

Sparsity is a default assumption these days.
Not always reasonable [Giannone et al. ’21, Econometrica] but still very used in “current hot topics” (e.g. interpretable AI).

Why it works?
- Solves loss of interpretability with fewer predictors
- We can tune prediction accuracy (bias-variance tradeoff) through “optimal model complexity”
- Partially Fix statistical issues (we go back to the regime $n<<p$)



## How to implement sparsity?

We need algorithm that select the best $k$ variables among those available?

How do I know these variable? How do I know $k$?

One route is called **best subset selection** ,

- Try all the models of size $k$ and compare them based on prediction.
- Try all possible size $k=1,\ldots,p$


 Without going into detail, I will just remark that this is a **really** hard and computationally demanding problem, so we typically go another route (although some alternatives exist: random subset, random projections,...).

 Also, this does not solve the $p>n$ problems. Penalised regression does

## Penalised regression

These *algorithms* use the *same linear models* we have seen before. But they use slightly different loss functions.

Recall that our methods so far focused on minimizing the square loss, _i.e_

$$\sum_{i=1}^n (y_i - \bx_i' \bb)^2$$

The class of algorithms we discuss now are **shrinkage methods**; they are based on adapting the loss function to *shrink* coefficients in a meaningful direction. It turns out shrinking towards 0 is often a good choice (think of the sparsity notion that we just introduced), and we look for parameters such that

$$ \bb^g = \arg\min_\bb \sum_{i=1}^n (y_i - \bx_i' \bb)^2 + \lambda \sum_{j=1}^{p} g(\beta_j)$$

where $g(\beta_j)$ is a **penalty** term, that penalizes $\beta_j$ by $\lambda\geq0$ when $\beta_j \neq  0$; recall that $\beta_j = 0$ means that feature $j$ (e.g., $j-1$ polynomial order, in the current case) is dropped from the model.

$\lambda$ is a key parameter (we will come back to it). Now, let's look at the different type of penalties.


## Types penalties

Procedures based on this type of loss functions are broadly called _penalized regression_.

Many different kind of penalties. Some "famous ones" - and the names the corresponding algorithms are known with:

+ LASSO ($L_1$ penalty): $g(\beta) = \sum_{i=1}^p |\beta_j|$

+ Ridge regression ($L_2$ penalty): $g(\beta) = \sum_{i=1}^p \beta_j^2$

+ Elastic Net: it is a comprimised between the two types of penalties

$$ g(\beta,\alpha) = \sum_{j=1}^p(\alpha|\beta_j| + (1-\alpha)\beta_j^2)$$

Let's look at these penalties

In [ ]:
beta  <- seq(-3,3,0.0001)
ridge_pen  <- beta^2
lasso_pen  <- abs(beta)
elnet_pen  <- 1/2*ridge_pen + 1/2*lasso_pen

plot(beta,ridge_pen,col=rgb(0,0,1/2,1),t='l',ylim=c(0,3))
lines(beta,lasso_pen,col='firebrick',lty=2,lwd=2)
lines(beta,elnet_pen,col='orange',lty=6,lwd=4)
legend('top',c('Ridge','LASSO','Elastic Net'),lty=c(1,2,6),lwd=3,col=c(rgb(0,0,1/2,1),'firebrick','orange'))

**Not all penalties sparse solutions**: LASSO aims to have some coefficients to $0$ (sparse), Ridge solely push them closer to $0$, but not equal to 0, Elastic net the first term encourages a sparse solution and the second encourages highly correlated features to be averaged.

+ The elastic net performs Shrinkage and Selection, and strikes a balance between the hard selection of the lasso and the averaging of similar coefficients of Ridge.

## Shrinkage vs Selection


All these models can be cast as restricted optimization problems. For instance, Ridge can be cast as
$$ \bb^{Ridge} = \arg\min_{\bb} \sum_{i=1}^n\Big( y_i - \bx_i' \bb \Big)^2$$
subject to  $$  \sum_{j=1}^{p} \beta_j^2 < t $$



Lasso can be cast as
$$ \bb^{LASSO} = \arg\min_{\bb} \sum_{i=1}^n\Big( y_i - \bx_i' \bb \Big)^2$$
subject to  $$  \sum_{j=1}^{p} |\beta_j| < t $$

In both cases, there is a one-to-one correspondence between $\lambda$ and $t$.

The two optimizations problems lead to different solutions:

<img src='https://github.com/barcelonagse-datascience/academic_files/raw/master/images/closedform_lassoridge.png'>


The picture above tells us
- LASSO is likely to set some coefficient to 0
- Ridge makes the coefficient smaller but keeps them not exactly equal

## Feature standardization:

a) Different coefficients are penalized in the same way: this only makes sense if the different coefficients have similar magnitudes.
        
b) Penalized likelihood algorithms require that the features have been standardized to have comparable scales. We often subtract the sample mean and divide by the standard deviation across replications

## Analytical solutions

These penalised problems do not admit analytical solutions, except for some particular cases

In case of orthonormal predictors, we have that the coefficients for the lasso are given by

$$ \hat \beta_j^{Ridge} = \hat \beta_j/(1 + \lambda), \hspace{2em} \text{ and } \hspace{2em} \hat \beta_j^{LASSO} = \text{sign}(\hat \beta_j)(|\hat \beta_j | - \lambda)_+ $$

<img src='https://github.com/barcelonagse-datascience/academic_files/raw/master/images/bias_lasso_ridge.png'>

So this means that

- Coefficients of penalised regression methods are **biased**
- Recall that OLS coefficients are (under assumptions) unbiased
- The bias is tuned by $\lambda$ and by the penalty chosen

## The role of $\lambda$:

Let's check how the solutions of LASSO and ridge change as we vary $\lambda$.

To do that, we load a more complex (still synthetic datasets). Don't worry at the LASSO/Ridge Implementation for now, we will get back to it


In [ ]:
data2 <- read.table("https://raw.githubusercontent.com/barcelonagse-datascience/academic_datasets/refs/heads/main/sim_data.csv", header = TRUE,
    sep = ",")
head(data2)

In [ ]:
#Ridge solution as we vary lambda
fit.ridge <- cv.glmnet(x=as.matrix(data2[,2:dim(data2)[2]]), y=data2[,1], alpha = 0, nfolds=10, lambda = exp(seq(from=log(1.001), to=log(20000), length.out = 100)))
plot(fit.ridge$glmnet.fit, xvar='lambda', main="Ridge coefficients")

In [ ]:
#Ridge solution as we vary lambda
fit.lasso <- cv.glmnet(x=as.matrix(data2[,2:dim(data2)[2]]), y=data2[,1], nfolds=10, lambda = exp(seq(from=log(0.0001), to=log(50), length.out = 100)))
plot(fit.lasso$glmnet.fit, xvar='lambda', main = 'LASSO coefficients')

Varying $\lambda$:

- we change the bias
- we change the model complexity (especially in the LASSO, some variables are dropped
- This has to do with the variance of my estimates: we effectively have more samples to estimate the parameters as I am making the model simpler



We can *tune* $\lambda$ to  
+ This **hyperparameter** (a.k.a *tuning parameter*) allows us to trade bias with variance, creating a continuum of mean squared errors along which we try to choose an optimal $\lambda$.
+ $\lambda \to 0$ leads to small bias/large variance (there is no penalty), $\lambda \to \infty$ to large bias/small variance (the model is effectively only estimating the constant term)
    + Ridge regression with varying $\lambda$s:
    <img src="https://github.com/barcelonagse-datascience/academic_files/raw/master/images/bias_variance_bishop.png" width="400">

## How do choose $\lambda$?

**Model with the best predictive performance**

We choose the balances with bias/variance, model complexity, etc. Choosing $\lambda$ that gives me the "best predictive model"

Two common rules
- choose $\lambda$ that minimize $MSE$ (recall the strategies to estimate it)
- choose $1se-\lambda$: it is the $\lambda$ that gives me the smaller model (more sparse, lower $k$) that is one-standard-errror away from the model that minimize $MSE$

### Lasso in action: the curve data with many many features

Let's go back to the original data set we have been looking at

In [ ]:
x_lasso  <- scale(data_large[,-which(colnames(data_large)%in% c('y','p0'))]) # scale non-intercept and non-predictor variables
x_lasso

The key package to do penalised regression in R is `glmnet`

In [ ]:
library(glmnet)

In [ ]:
lasso  <- glmnet(x=x_lasso,y=y,alpha=1) #alpha decides which penalty to use

In [ ]:
lasso

In [ ]:
lasso$beta

In [ ]:
plot(lasso)

In [ ]:
cbind('Lambda0'=lasso$beta[,1],
      'Lambda1'=lasso$beta[,10],
      'Lambda2'=lasso$beta[,50],
      'Lambda3'=lasso$beta[,75])

In [ ]:
ridge <- glmnet(x=x_lasso,y=y,alpha=0)

In [ ]:
plot(ridge)
abline(h=0)
grid()

In [ ]:
# Cross-validated lasso:
lasso_cv  <- cv.glmnet(x=x_lasso,y=y,alpha=1,nfolds=10,grouped=FALSE)

In [ ]:
plot(lasso_cv)

In [ ]:
lasso_coef  <- as.matrix(coef(lasso_cv, s='lambda.min'))
lasso_coef

In [ ]:
lasso_cv_pred  <- predict(lasso_cv, s='lambda.min',newx=x_lasso)
lasso_cv_pred

In [ ]:
par(mfrow=c(1,4))
plot(y,cvp_pred,pch=19,col='darkblue',cex=1.5,ylab='fitted values',xlab='realized values',
    ,main=sprintf('p=1, LOOCV R2: %.2f',cor(y,cvp_pred)^2))
lines(y,y,lwd=3,col='firebrick')
plot(y,cvp_med_pred,pch=19,col='darkblue',cex=1.5,ylab='fitted values',xlab='realized values'
     ,main=sprintf('p=3, LOOCV R2: %.2f',cor(y,cvp_med_pred)^2))
lines(y,y,lwd=3,col='firebrick')
plot(y,cvp_large_pred,pch=19,col='darkblue',cex=1.5,ylab='fitted values',xlab='realized values'
     ,main=sprintf('p=9, LOOCV R2: %.2f',cor(y,cvp_large_pred)^2))
lines(y,y,lwd=3,col='firebrick')
plot(y,lasso_cv_pred,pch=19,col='darkblue',cex=1.5,ylab='fitted values',xlab='realized values'
     ,main=sprintf('LASSO, LOOCV R2: %.2f',cor(y,lasso_cv_pred)^2))
lines(y,y,lwd=3,col='firebrick')

In [ ]:
x_scaled  <- scale(x_large_test)
x_scaled[,1]  <- 1
x_scaled

In [ ]:
y_hat_lasso  <- x_scaled %*% lasso_coef

In [ ]:
plot(x_large_test$p1,y_hat_lasso,t='l',col='firebrick',lwd=2,ylim=c(-1,1.4), ylab='y',xlab='x')
points(data,col='orange',pch=19,cex=1.5)
grid()
legend('topright',c('Training Data','Test Data'), col = c('orange','firebrick'),
       lty=c(NA,1), pch=c(19,NA),lwd=4)

## Feature selection

It will be very tempting to exploit the fact that the LASSO drop some variables and keep others to try to interpret which variables are the most relevant to predict something and start from there an attempt to draw associations. As a general rule of thumb, *it is better not to do it*!

- there are methods that have better properties to perform feature selection
- All the parameters chosen are driving by prediction, not by feature selection

It is anyway possible to have a look at the non-zero coefficients


In [ ]:
coef(lasso_cv)

A cautionary example.

The true model is
$$ Y= \beta X_1 + \epsilon$$
but we observe $(X_1,X_2,X_3)$, with $cor(X_1,X_2)=cor(X_1,X_3)=cor(X_2,X_3)=0.98$, so we decide to fit the following linear regression
$$ Y= \beta_1 X_1+\beta_2 X_2+\beta_3 X_3 + \epsilon$$
Which variable will LASSO select?

### Further insights & observations on LASSO

+ Sparsity: increasing values of $\lambda$ have the effect that an increasing number of estimated coefficients are exactly zero


+ Convexity: the loss function is convex; this is because the least squares function is convex (a quadratic function) and the penalty is convex too. This allows very efficient estimation using **convex optimization** algorithms.
    + A common choice is **coordinate-wise descent**. This is an iterative algorithm that scans through each coefficient and updates it using information about the values of all other coefficients.
    + For standardized features $\bx_1,\bx_2,\ldots$ each coefficient is updated as:
$$\beta_j \leftarrow \mathcal{S}_{\lambda}\left({1 \over n} \boldsymbol{r}_{-j}^T \bx_j\right )$$
where $\boldsymbol{r}_{-j}$ is the vector of residuals from the model with $\beta_j = 0$ and the soft-thresholding operator is:
      $$\mathcal{S}_{\lambda}(\beta) = \mathrm{sign}(\beta) \max\{|\beta| - \lambda,0\}$$
    + The fast optimization is a major attraction for the lasso
      + Coordinate-wise descent is implemented at a cost that grows only linearly in $n$ and $p$: it is a practical solution for Big Data and Big Models  

## Some other details


+ Building good predictive models with hundreds or even thousands of features is a real possibility
+ LASSO combines least squares with a penalty for model complexity; it relies on an additional *regularization parameter*
+ The choice of regularization hyperparameter is a model selection problem; you can use both cross validation to estimate the MSE for each possible value of the hyperparameter and use a grid search to identify good values for the hyperparameter.
+ **Inference with the output of the lasso model is non-trivial** and subject of more advanced material. Although lasso implicitly selects a model by dropping variables, you should not over-interpret the variables that have been selected. Its merit is primary in getting a good predictive model. Lasso is helpful in screening some variables, so it is often used as a first step to be followed by a more formal selection procedure. Generally, these questions fall under the theme of *post-selection* inference

## Exporting to latex

In [ ]:
library(xtable)
table  <- xtable(reg)
lines  <- print(table, sanitize.text.function = identity,booktab=TRUE)
lines  <- strsplit(lines,'\n')[[1]]
lines

## References

James, G., Witten, D., Hastie, T., & Tibshirani, R. (2021). *An introduction to statistical learning: with applications in R*. New York: springer. Chapter 1,2, 3 (section 3.1.2 & 3.1.3 not strictly necessary), Chapter 5 (skip 5.2) https://www.statlearning.com/

*More advanced material:*

Hastie, T., Tibshirani, R., Friedman, J., 2009. *Elements of Statistical Learning*. 2nd Edition. Chapters 1, 2, 3. Section 3.4; More advanced 3.8,3.9,7.10  https://web.stanford.edu/~hastie/ElemStatLearn/

Bishop, C.M. *Pattern recognition and machine learning*. Chapter 1, Sections 3.1, 3.2